# Prosodic-Based Vishing Detector

## Imports and Configurations

In [ ]:
from pathlib import Path
import sys
import os
import hashlib

import numpy as np
import pandas as pd
import joblib
import torch
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import TensorDataset, DataLoader
from joblib import Parallel, delayed
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib

from sklearn.preprocessing import RobustScaler

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from modules.dataset_processing import extract_archives, prepare_all_split_dataframes, subsample_by_class
from modules.audio_processing import extract_prosodic_features, PROSODIC_FEATURE_COLUMNS
from modules.models import ProsodyMLP, ProsodyMLPConfig
from modules.attacks import FGSMAttack, FGSMAttackConfig

In [ ]:
DATA_ROOT = PROJECT_ROOT / 'ASVspoof5'
MODEL_DIR = PROJECT_ROOT / 'models'
CHECKPOINT_DIR = MODEL_DIR / 'prosodic'
CACHE_DIR = DATA_ROOT / 'feature_cache'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

TEST_MODE = False
BONAFIDE_SUBSAMPLE = 15000
SPOOF_SUBSAMPLE = 15000
BATCH_SIZE = 256
NUM_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 15
MIN_DELTA = 1e-4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-3
RANDOM_SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_JOBS = max(1, os.cpu_count() - 2)

torch.manual_seed(RANDOM_SEED)
print({'device': str(DEVICE), 'data_root': str(DATA_ROOT), 'checkpoint_dir': str(CHECKPOINT_DIR)})

## Data Preprocessing

In [ ]:
extract_archives(DATA_ROOT)
split_frames = prepare_all_split_dataframes(DATA_ROOT, require_existing_files=True)

if TEST_MODE:
    split_frames['train'] = subsample_by_class(split_frames['train'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)
    split_frames['dev'] = subsample_by_class(split_frames['dev'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)
    split_frames['eval'] = subsample_by_class(split_frames['eval'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)

for split_name, frame in split_frames.items():
    print(split_name, frame.shape, frame['LABEL'].value_counts().to_dict())

## Prosodic Feature Extraction

In [ ]:
import shutil

CLEAR_FEATURE_CACHE = False  # Set to True to force rebuild, False to reuse

if CLEAR_FEATURE_CACHE and CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Cleared feature cache at {CACHE_DIR}")
elif not CACHE_DIR.exists():
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Created feature cache at {CACHE_DIR}")
else:
    print(f"Using existing cache at {CACHE_DIR}")

print(f"Feature count: {len(PROSODIC_FEATURE_COLUMNS)}")

In [ ]:
def get_features(file_path):
    safe_name = file_path.replace("/", "_").replace(":", "_")
    cache_path = CACHE_DIR / (safe_name + ".pkl")
    if cache_path.exists():
        return joblib.load(str(cache_path))
    # Fallback: compute and save if missing
    feats = extract_prosodic_features(file_path)
    joblib.dump(feats, str(cache_path))
    return feats


def extract_one(row):
    feats = get_features(row["FULL_FILE_PATH"])
    feats["ATTACK_LABEL"] = row["ATTACK_LABEL"]
    feats["LABEL"] = row["LABEL"]
    return feats


def build_feature_df(df, split_name):
    rows = [row for _, row in df.iterrows()]
    with tqdm_joblib(tqdm(desc=f"Extracting features for {split_name}", total=len(rows))):
        records = Parallel(n_jobs=N_JOBS)(delayed(extract_one)(row) for row in rows)
    return pd.DataFrame(records)

train_feats = build_feature_df(split_frames['train'], 'train')
dev_feats = build_feature_df(split_frames['dev'], 'dev')
eval_feats = build_feature_df(split_frames['eval'], 'eval')

In [ ]:
bonafide_mask = train_feats['LABEL'] == 0

print("=== Feature Separation Check (Train) ===")
for col in ['jitter', 'shimmer', 'harmonics_to_noise_ratio', 'f0_std', 'voiced_ratio',
            'f1_mean', 'f2_mean', 'f0_mean', 'spectral_flatness_mean', 'pause_ratio']:
    b_mean = train_feats.loc[bonafide_mask, col].mean()
    s_mean = train_feats.loc[~bonafide_mask, col].mean()
    ratio = abs(b_mean - s_mean) / (train_feats[col].std() + 1e-8)
    print(f"{col:<35} bonafide={b_mean:.4f}  spoof={s_mean:.4f}  separation={ratio:.3f}")

In [ ]:
# Build scaled feature matrices

for df in [train_feats, dev_feats, eval_feats]:
    df[PROSODIC_FEATURE_COLUMNS] = df[PROSODIC_FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)

train_medians = train_feats[PROSODIC_FEATURE_COLUMNS].median()

for df_name, df in [("train", train_feats), ("dev", dev_feats), ("eval", eval_feats)]:
    df[PROSODIC_FEATURE_COLUMNS] = df[PROSODIC_FEATURE_COLUMNS].fillna(train_medians)

X_train = train_feats[PROSODIC_FEATURE_COLUMNS].values.astype(np.float32)
y_train = train_feats["LABEL"].values.astype(np.int64)

X_dev = dev_feats[PROSODIC_FEATURE_COLUMNS].values.astype(np.float32)
y_dev = dev_feats["LABEL"].values.astype(np.int64)

X_eval = eval_feats[PROSODIC_FEATURE_COLUMNS].values.astype(np.float32)
y_eval = eval_feats["LABEL"].values.astype(np.int64)

scaler = RobustScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_dev = scaler.transform(X_dev).astype(np.float32)
X_eval = scaler.transform(X_eval).astype(np.float32)
joblib.dump(scaler, CHECKPOINT_DIR / 'scaler.joblib')

train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
dev_dataset   = TensorDataset(torch.tensor(X_dev),   torch.tensor(y_dev))
eval_dataset  = TensorDataset(torch.tensor(X_eval),  torch.tensor(y_eval))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
eval_loader  = DataLoader(eval_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

## Training

In [ ]:
model_config = ProsodyMLPConfig(
    input_dim=len(PROSODIC_FEATURE_COLUMNS),
    num_classes=2,               # was 9
    hidden_dims=(256, 128, 64),
    dropout=0.45,
    noise_std=0.03,
    use_attention=True,
)
model = ProsodyMLP(config=model_config).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)

num_bonafide = (split_frames['train']['LABEL'] == 0).sum()
num_spoof = (split_frames['train']['LABEL'] == 1).sum()
ratio = num_spoof / num_bonafide  # true data should be ~9.0
class_weights = torch.tensor([3.0, 1.0], dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

model

In [ ]:
from sklearn.metrics import roc_curve
import numpy as np



def compute_eer_from_loader(model, dataloader):
    model.eval()
    all_labels = []
    all_bonafide_probs = []

    with torch.no_grad():
        for features, labels in dataloader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(features)
            probs = torch.softmax(logits, dim=1)
            all_bonafide_probs.append(probs[:, 0].cpu())
            all_labels.append(labels.cpu())

    all_labels = torch.cat(all_labels).numpy()
    all_bonafide_probs = torch.cat(all_bonafide_probs).numpy()

    bonafide_labels = (all_labels == 0).astype(int)
    fpr, tpr, _ = roc_curve(bonafide_labels, all_bonafide_probs)
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2
    return eer


def run_epoch(model, dataloader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_examples = 0

    for features, labels in dataloader:
        features = features.to(DEVICE)
        labels = labels.to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

        logits = model(features)
        loss = criterion(logits, labels)

        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # added
            optimizer.step()

        total_examples += labels.shape[0]
        total_loss += loss.item() * labels.shape[0]

    return total_loss / max(total_examples, 1)


best_dev_loss = float('inf')
best_dev_eer = float('inf')
best_loss_epoch = 0
best_eer_epoch = 0
epochs_without_improvement = 0
best_checkpoint_path = CHECKPOINT_DIR / 'best_loss.pt'
best_eer_checkpoint_path = CHECKPOINT_DIR / 'best_eer.pt'
latest_checkpoint_path = CHECKPOINT_DIR / 'latest.pt'

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer=optimizer)
    dev_loss = run_epoch(model, dev_loader, optimizer=None)
    scheduler.step(dev_loss)

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'dev_loss': dev_loss,
    }

    # Save latest (removed per-epoch saves)
    torch.save(checkpoint, latest_checkpoint_path)

    # Best loss model
    improved = dev_loss < (best_dev_loss - MIN_DELTA)
    if improved:
        best_dev_loss = dev_loss
        best_loss_epoch = epoch
        torch.save(checkpoint, best_checkpoint_path)
        print(f"→ New best loss model saved at epoch {epoch} (dev_loss: {dev_loss:.5f})")


    dev_eer = compute_eer_from_loader(model, dev_loader)

    if dev_eer < best_dev_eer:
        best_dev_eer = dev_eer
        best_eer_epoch = epoch
        epochs_without_improvement = 0
        eer_checkpoint = {**checkpoint, 'best_dev_eer': best_dev_eer}
        torch.save(eer_checkpoint, best_eer_checkpoint_path)
        print(f"→ New best EER model saved at epoch {epoch} (EER: {best_dev_eer*100:.2f}%)")
    else:
        epochs_without_improvement += 1

    current_lr = optimizer.param_groups[0]['lr']
    print({
        'epoch': epoch,
        'train_loss': round(train_loss, 5),
        'dev_loss': round(dev_loss, 5),
        'dev_eer': f"{dev_eer*100:.2f}%" if dev_eer is not None else "—",
        'best_dev_eer': f"{best_dev_eer*100:.2f}%" if best_dev_eer != float('inf') else "—",
        'lr': current_lr,
        'best_loss_epoch': best_loss_epoch,
        'best_eer_epoch': best_eer_epoch,
        'no_improve': epochs_without_improvement,
    })

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch} (best EER epoch: {best_eer_epoch})")
        break

In [ ]:
# Load best checkpoint and save scaler
best_checkpoint = torch.load(best_checkpoint_path, weights_only=False)
model.load_state_dict(best_checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {best_checkpoint['epoch']}, saved scaler")

## FGSM Evaluation (Prosodic Features)

In [ ]:
feature_scale = float(np.mean(np.std(X_train, axis=0)))
fgsm_epsilon = max(1e-4, 0.05 * feature_scale)
clamp_min = float(np.percentile(X_train, 1))
clamp_max = float(np.percentile(X_train, 99))

fgsm_attack = FGSMAttack(
    config=FGSMAttackConfig(
        epsilon=fgsm_epsilon,
        clamp_min=clamp_min,
        clamp_max=clamp_max,
        targeted=False,
    )
)

print({
    'epsilon': round(fgsm_epsilon, 6),
    'clamp_min': round(clamp_min, 4),
    'clamp_max': round(clamp_max, 4),
})

In [ ]:
all_labels = []
all_clean_predictions = []
all_adversarial_predictions = []
all_clean_true_probabilities = []
all_adversarial_true_probabilities = []

model.eval()
for features, labels in eval_loader:
    features = features.to(DEVICE)
    labels = labels.to(DEVICE)

    with torch.no_grad():
        clean_logits = model(features)
        clean_probabilities = torch.softmax(clean_logits, dim=1)
        clean_predictions = clean_probabilities.argmax(dim=1)
        clean_true_probabilities = clean_probabilities[:, 0]

    attack_result = fgsm_attack.generate(model=model, inputs=features, labels=labels)
    adversarial_features = attack_result.adversarial_inputs

    with torch.no_grad():
        adversarial_logits = model(adversarial_features)
        adversarial_probabilities = torch.softmax(adversarial_logits, dim=1)
        adversarial_predictions = adversarial_probabilities.argmax(dim=1)
        adversarial_true_probabilities = adversarial_probabilities[:, 0]

    all_labels.append(labels.detach().cpu())
    all_clean_predictions.append(clean_predictions.detach().cpu())
    all_adversarial_predictions.append(adversarial_predictions.detach().cpu())
    all_clean_true_probabilities.append(clean_true_probabilities.detach().cpu())
    all_adversarial_true_probabilities.append(adversarial_true_probabilities.detach().cpu())

all_labels = torch.cat(all_labels)
all_clean_predictions = torch.cat(all_clean_predictions)
all_adversarial_predictions = torch.cat(all_adversarial_predictions)
all_clean_true_probabilities = torch.cat(all_clean_true_probabilities)
all_adversarial_true_probabilities = torch.cat(all_adversarial_true_probabilities)

print({'num_samples': int(all_labels.shape[0])})

### Accuracy

### Mean Squared Confidence Degradation (MSCD)

### Attack Success Rate (ASR)

In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

def compute_eer(labels, scores):
    """
    labels: 0 = bonafide, 1 = spoof
    scores: model's probability of being BONAFIDE (class 0)
    EER is where FAR == FRR
    """
    # sklearn's roc_curve treats 1 as positive
    # We want bonafide (0) as the "positive" class for ASV context
    # So flip: use probability of bonafide, and flip labels
    bonafide_scores = scores  # probability assigned to bonafide class
    bonafide_labels = (labels == 0).astype(int)

    fpr, tpr, thresholds = roc_curve(bonafide_labels, bonafide_scores)
    
    fnr = 1 - tpr  # False Negative Rate = FRR
    # EER is where FPR (FAR) == FNR (FRR)
    eer_index = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[eer_index] + fnr[eer_index]) / 2
    eer_threshold = thresholds[eer_index]
    
    return eer, eer_threshold

In [ ]:
clean_accuracy = (all_clean_predictions == all_labels).float().mean().item()
adversarial_accuracy = (all_adversarial_predictions == all_labels).float().mean().item()

print(f"Clean Accuracy: {clean_accuracy:.4f}")
print(f"Adversarial Accuracy: {adversarial_accuracy:.4f}")

delta_probabilities = all_clean_true_probabilities - all_adversarial_true_probabilities
mscd = (delta_probabilities ** 2).mean().item()

print(f"MSCD: {mscd:.6f}")

asr = (all_clean_predictions != all_adversarial_predictions).float().mean().item()

print(f"ASR: {asr:.4f}")

# Convert to numpy for sklearn
labels_np = all_labels.numpy()
clean_probs_np = all_clean_true_probabilities.numpy()
adversarial_probs_np = all_adversarial_true_probabilities.numpy()


print(f"Score range: clean=[{clean_probs_np.min():.3f}, {clean_probs_np.max():.3f}]  "
      f"adv=[{adversarial_probs_np.min():.3f}, {adversarial_probs_np.max():.3f}]")
print(f"Label dist: bonafide={int((labels_np==0).sum())}  spoof={int((labels_np==1).sum())}")

# Clean EER
clean_eer, clean_threshold = compute_eer(labels_np, clean_probs_np)
print(f"Clean EER:       {clean_eer * 100:.2f}%  (threshold: {clean_threshold:.4f})")

# Adversarial EER
adv_eer, adv_threshold = compute_eer(labels_np, adversarial_probs_np)
print(f"Adversarial EER: {adv_eer * 100:.2f}%  (threshold: {adv_threshold:.4f})")

In [ ]:
del all_labels, all_clean_predictions, all_adversarial_predictions
del all_clean_true_probabilities, all_adversarial_true_probabilities
import gc; gc.collect()

In [ ]:
# Load best checkpoint and save scaler
best_checkpoint = torch.load(best_eer_checkpoint_path, weights_only=False)
model.load_state_dict(best_checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {best_checkpoint['epoch']}, saved scaler")

In [ ]:
feature_scale = float(np.mean(np.std(X_train, axis=0)))
fgsm_epsilon = max(1e-4, 0.05 * feature_scale)
clamp_min = float(np.percentile(X_train, 1))
clamp_max = float(np.percentile(X_train, 99))

fgsm_attack = FGSMAttack(
    config=FGSMAttackConfig(
        epsilon=fgsm_epsilon,
        clamp_min=clamp_min,
        clamp_max=clamp_max,
        targeted=False,
    )
)

print({
    'epsilon': round(fgsm_epsilon, 6),
    'clamp_min': round(clamp_min, 4),
    'clamp_max': round(clamp_max, 4),
})

In [ ]:
all_labels = []
all_clean_predictions = []
all_adversarial_predictions = []
all_clean_true_probabilities = []
all_adversarial_true_probabilities = []

model.eval()
for features, labels in eval_loader:
    features = features.to(DEVICE)
    labels = labels.to(DEVICE)

    with torch.no_grad():
        clean_logits = model(features)
        clean_probabilities = torch.softmax(clean_logits, dim=1)
        clean_predictions = clean_probabilities.argmax(dim=1)
        clean_true_probabilities = clean_probabilities[:, 0]

    attack_result = fgsm_attack.generate(model=model, inputs=features, labels=labels)
    adversarial_features = attack_result.adversarial_inputs

    with torch.no_grad():
        adversarial_logits = model(adversarial_features)
        adversarial_probabilities = torch.softmax(adversarial_logits, dim=1)
        adversarial_predictions = adversarial_probabilities.argmax(dim=1)
        adversarial_true_probabilities = adversarial_probabilities[:, 0]

    all_labels.append(labels.detach().cpu())
    all_clean_predictions.append(clean_predictions.detach().cpu())
    all_adversarial_predictions.append(adversarial_predictions.detach().cpu())
    all_clean_true_probabilities.append(clean_true_probabilities.detach().cpu())
    all_adversarial_true_probabilities.append(adversarial_true_probabilities.detach().cpu())

all_labels = torch.cat(all_labels)
all_clean_predictions = torch.cat(all_clean_predictions)
all_adversarial_predictions = torch.cat(all_adversarial_predictions)
all_clean_true_probabilities = torch.cat(all_clean_true_probabilities)
all_adversarial_true_probabilities = torch.cat(all_adversarial_true_probabilities)

print({'num_samples': int(all_labels.shape[0])})

In [ ]:
clean_accuracy = (all_clean_predictions == all_labels).float().mean().item()
adversarial_accuracy = (all_adversarial_predictions == all_labels).float().mean().item()

print(f"Clean Accuracy: {clean_accuracy:.4f}")
print(f"Adversarial Accuracy: {adversarial_accuracy:.4f}")

delta_probabilities = all_clean_true_probabilities - all_adversarial_true_probabilities
mscd = (delta_probabilities ** 2).mean().item()

print(f"MSCD: {mscd:.6f}")

asr = (all_clean_predictions != all_adversarial_predictions).float().mean().item()

print(f"ASR: {asr:.4f}")

# Convert to numpy for sklearn
labels_np = all_labels.numpy()
clean_probs_np = all_clean_true_probabilities.numpy()
adversarial_probs_np = all_adversarial_true_probabilities.numpy()


print(f"Score range: clean=[{clean_probs_np.min():.3f}, {clean_probs_np.max():.3f}]  "
      f"adv=[{adversarial_probs_np.min():.3f}, {adversarial_probs_np.max():.3f}]")
print(f"Label dist: bonafide={int((labels_np==0).sum())}  spoof={int((labels_np==1).sum())}")

# Clean EER
clean_eer, clean_threshold = compute_eer(labels_np, clean_probs_np)
print(f"Clean EER:       {clean_eer * 100:.2f}%  (threshold: {clean_threshold:.4f})")

# Adversarial EER
adv_eer, adv_threshold = compute_eer(labels_np, adversarial_probs_np)
print(f"Adversarial EER: {adv_eer * 100:.2f}%  (threshold: {adv_threshold:.4f})")
print("res")